# Load data to Delta tables 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType, FloatType
import pyspark.sql.functions as f

## Define schemas for every table 

In [0]:
catalog_name = "brazilian_e_commerce"
volume_path = "/Volumes/brazilian_e_commerce/row_data/row"


In [0]:
files = dbutils.fs.ls(volume_path)

csv_files = [file for file in files if file.path.endswith('.csv')]

display(csv_files)

path,name,size,modificationTime
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1776451262000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1776451263000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1776451264000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1776451265000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1776451266000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1776451267000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1776451267000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1776451267000
dbfs:/Volumes/brazilian_e_commerce/row_data/row/product_category_name_translation.csv,product_category_name_translation.csv,2613,1776451267000


In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat", FloatType(), True),
    StructField("geolocation_lng", FloatType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True)
])

order_items_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), False),
    StructField("seller_id", StringType(), False),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("freight_value", FloatType(), True)
])

payments_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", FloatType(), True)
])

reviews_schema = StructType([
    StructField("review_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", StringType(), True),
    StructField("review_answer_timestamp", StringType(), True)
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", FloatType(), True),
    StructField("product_height_cm", FloatType(), True),
    StructField("product_width_cm", FloatType(), True)
])

sellers_schema = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True)
])

product_category_name_translation_schema = StructType([
    StructField("product_category_name", StringType(), False),
    StructField("product_category_name_english", StringType(), True)
])



## Create Dataframe and save to tables with metadata 

In [0]:
tables_config = {
    "customers": {
        "file": "olist_customers_dataset.csv", 
        "schema": customers_schema
    },
    "geolocation": {
        "file": "olist_geolocation_dataset.csv", 
        "schema": geolocation_schema
    },
    "order_items": {
        "file": "olist_order_items_dataset.csv", 
        "schema": order_items_schema
    },
    "order_payments": {
        "file": "olist_order_payments_dataset.csv", 
        "schema": payments_schema
    },
    "order_reviews": {
        "file": "olist_order_reviews_dataset.csv", 
        "schema": reviews_schema
    },
    "orders": {
        "file": "olist_orders_dataset.csv", 
        "schema": orders_schema
    },
    "products": {
        "file": "olist_products_dataset.csv", 
        "schema": products_schema
    },
    "sellers": {
        "file": "olist_sellers_dataset.csv", 
        "schema": sellers_schema
    },
    "product_category_name_translation": {
        "file": "product_category_name_translation.csv", 
        "schema": product_category_name_translation_schema
    }
}

In [0]:
for table_name, config in tables_config.items():
    # Build paths
    file_path = f"{volume_path}/{config['file']}"
    current_schema = config["schema"]
    
    # Read CSV to Spark DataFrame
    df_raw = spark.read \
        .option("header", "true") \
        .option("delimiter", ",") \
        .option("multiLine", "true") \
        .option("quote", '"') \
        .option("escape", '"') \
        .schema(current_schema) \
        .csv(file_path)

    # Add metadata to tables
    df_raw = df_raw \
        .withColumn("_ingested_at", f.current_timestamp()) \
        .withColumn("_source_file", f.col("_metadata.file_path"))
    
    # Save DataFrame as Delta table
    target_table = f"{catalog_name}.bronze.brz_{table_name}"
    
    df_raw.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(target_table)